# Using 🤗 PEFT & bitsandbytes to finetune a LoRa checkpoint




In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
# os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
!pip install -q bitsandbytes datasets accelerate loralib

In [ ]:

# !pip install -q git+https://github.com/huggingface/transformers.git@main git+https://github.com/huggingface/peft.git

In [ ]:
!pip uninstall -y peft transformers
!pip install -q peft transformers

In [ ]:
!nvidia-smi -L

In [ ]:
import random

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, LlamaTokenizer

In [ ]:
# traindata1 = load_dataset('ptb_text_only', 'penn_treebank')
# traindata2 = load_dataset('wikitext', 'wikitext-2-raw-v1')

In [ ]:
traindata3 = load_dataset('allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'})

In [ ]:
def get_tokenizer(model):

    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    return tokenizer

In [ ]:
orig_data=traindata3['train']['text'][:36718]
num=0.7*len(orig_data)
tokenizer=get_tokenizer('facebook/opt-125m')


In [ ]:
trainenc=tokenizer("\n".join(orig_data[:int(num)]), return_tensors='pt')
print('train data')


In [ ]:
testenc=tokenizer("\n".join(orig_data[int(num):]), return_tensors='pt')
print('test_data')

In [ ]:



# def set_seed(seed):
#     np.random.seed(seed)
#     torch.random.manual_seed(seed)



# def get_wikitext2(nsamples, seed, seqlen, model, tokenizer):

#     traindata = load_dataset('wikitext', 'wikitext-2-raw-v1')
#     # traindata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
#     # testdata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')

#     trainenc = tokenizer("\n\n".join(traindata['text']), return_tensors='pt',padding=True, truncation=True)
#     # testenc = tokenizer("\n\n".join(testdata['text']), return_tensors='pt')

#     # random.seed(seed)
#     # trainloader = []
#     # for _ in range(nsamples):
#     #     i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
#     #     j = i + seqlen
#     #     inp = trainenc.input_ids[:, i:j]
#     #     tar = inp.clone()
#     #     tar[:, :-1] = -100
#     #     trainloader.append((inp, tar))
#     # return trainloader, testenc
#     # return traindata,testdata
#     return trainenc,testenc

# def get_ptb(nsamples, seed, seqlen, model, tokenizer):
#     traindata = load_dataset('ptb_text_only', 'penn_treebank')
#     # traindata = load_dataset('ptb_text_only', 'penn_treebank', split='train')
#     # testdata = load_dataset('ptb_text_only', 'penn_treebank', split='test')

#     trainenc = tokenizer("\n".join(traindata['sentence']), return_tensors='pt')
#     # testenc = tokenizer("\n".join(testdata['sentence']), return_tensors='pt')

#     # random.seed(seed)
#     # trainloader = []
#     # for _ in range(nsamples):
#     #     i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
#     #     j = i + seqlen
#     #     inp = trainenc.input_ids[:, i:j]
#     #     tar = inp.clone()
#     #     tar[:, :-1] = -100
#     #     trainloader.append((inp, tar))
#     return trainenc, testenc

# def get_c4(nsamples, seed, seqlen, model, tokenizer):
#     traindata = load_dataset(
#         'allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train'
#     )
#     valdata = load_dataset(
#         'allenai/c4', data_files={'validation': 'en/c4-validation.00000-of-00008.json.gz'}, split='validation'
#     )

#     random.seed(seed)
#     trainloader = []
#     for _ in range(nsamples):
#         while True:
#             i = random.randint(0, len(traindata) - 1)
#             trainenc = tokenizer(traindata[i]['text'], return_tensors='pt')
#             if trainenc.input_ids.shape[1] > seqlen:
#                 break
#         i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
#         j = i + seqlen
#         inp = trainenc.input_ids[:, i:j]
#         tar = inp.clone()
#         tar[:, :-1] = -100
#         trainloader.append((inp, tar))

#     valenc = tokenizer(' '.join(valdata[:1100]['text']), return_tensors='pt')
#     valenc = valenc.input_ids[:, :(256 * seqlen)]

#     class TokenizerWrapper:
#         def __init__(self, input_ids):
#             self.input_ids = input_ids
#     valenc = TokenizerWrapper(valenc)

#     return trainloader, valenc

# def get_loaders(name, nsamples=128, seed=0, seqlen=2048, model=''):
#     tokenizer = get_tokenizer(model)
#     if 'wikitext2' in name:
#         return get_wikitext2(nsamples, seed, seqlen, model, tokenizer)
#     if 'ptb' in name:
#         return get_ptb(nsamples, seed, seqlen, model, tokenizer)
#     if 'c4' in name:
#         return get_c4(nsamples, seed, seqlen, model, tokenizer)


In [ ]:
name="facebook/opt-125m"
# for dataset in ['wikitext2']:
#     trainenc, testenc = get_loaders(
#         dataset, seed=0, model=name, seqlen=2048
#     )
#     print(dataset)

In [ ]:
# testenc = testenc.input_ids
import math
from datasets import Dataset
mseq=2048
nsamples = math.floor(trainenc.input_ids.numel() // mseq)
data=[]
for i in range(nsamples):
        data.append( {
        "input_ids": trainenc.input_ids[:, (i * mseq):((i + 1) * mseq)][0],
        "attention_mask": trainenc.attention_mask[:, (i * mseq):((i + 1) * mseq)][0]
    })


dataset = Dataset.from_list(data)

### Setup the model

In [ ]:
# torch.cuda.empty_cache()
# del model

In [ ]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch
import torch.nn as nn
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

name="facebook/opt-125m"
model = AutoModelForCausalLM.from_pretrained(
    name,
    # load_in_8bit=True,
    # device_map='auto',
)

tokenizer = AutoTokenizer.from_pretrained(name)

In [ ]:
layers_to_delete=[3,4,7]
total_layers=len(model.model.decoder.layers)
for i in sorted(layers_to_delete, reverse=True):
    if i>=0 or i<len(model.model.decoder.layers):
        del model.model.decoder.layers[i]

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [ ]:
print(f"Sparsity {len(layers_to_delete)/total_layers}")

### Freezing the original weights


In [ ]:
for param in model.parameters():
  param.requires_grad = False  # freeze the model - train adapters later
  if param.ndim == 1:
    # cast the small parameters (e.g. layernorm) to fp32 for stability
    param.data = param.data.to(torch.float32)

model.gradient_checkpointing_enable()  # reduce number of stored activations
model.enable_input_require_grads()

class CastOutputToFloat(nn.Sequential):
  def forward(self, x): return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)

### Setting up the LoRa Adapters

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16, #attention heads
    lora_alpha=32, #alpha scaling
    # target_modules=["q_proj", "v_proj"], #if you know the
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # set this for CLM or Seq2Seq
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

## Data

In [ ]:
import transformers
from datasets import load_dataset
# data = load_dataset("Abirate/english_quotes")


### Training

In [ ]:
def add_labels(example):
    example["labels"] = example["input_ids"]
    return example

dataset = dataset.map(add_labels)


In [ ]:
dataset

In [ ]:

trainer = transformers.Trainer(
    model=model,
    # train_dataset=data['train'],
    train_dataset=dataset,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        max_steps=50,
        # learning_rate=2e-4,
        learning_rate=0.001,
        fp16=True,
        logging_steps=1,
        # output_dir='outputs'
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

In [ ]:
for name, param in model.named_parameters():

    param.requires_grad = False  # Freeze the parameter (no gradients)


In [ ]:
import torch
import torch.nn as nn
import copy

def opt_eval(model, testenc, dev, dataset: str, log_wandb: bool = False):
    print('Evaluating ...')

    mseq = 2048  # Model sequence length
    testenc = testenc.input_ids
    nsamples = testenc.numel() // mseq
    # model=model.base_model
    use_cache = model.config.use_cache
    model.config.use_cache = False
    # model.base_model.model.model.decoder.layers
    layers = model.model.decoder.layers  # Get model layers

    # Move necessary components to GPU
    model.model.decoder.embed_tokens = model.model.decoder.embed_tokens.to(dev)
    model.model.decoder.embed_positions = model.model.decoder.embed_positions.to(dev)

    if hasattr(model.model.decoder, 'project_out') and model.model.decoder.project_out:
        model.model.decoder.project_out = model.model.decoder.project_out.to(dev)
    if hasattr(model.model.decoder, 'project_in') and model.model.decoder.project_in:
        model.model.decoder.project_in = model.model.decoder.project_in.to(dev)

    # Create buffer to store activations
    dtype = next(model.parameters()).dtype
    # inps = torch.zeros((nsamples, mseq, model.config.hidden_size), dtype=dtype, device=dev)
    inps = torch.zeros((nsamples, mseq, model.config.hidden_size), dtype=dtype)
    cache = {'i': 0, 'attention_mask': None}

    # Catcher to intercept first-layer activations
    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp  # Store first-layer hidden state
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            raise ValueError  # Stop execution

    original_layer=copy.deepcopy(layers[0])

    layers[0] = Catcher(layers[0])  # Wrap first layer with Catcher

    # Run first layer to capture activations
    for i in range(nsamples):
        batch = testenc[:, (i * mseq):((i + 1) * mseq)].to(dev)
        try:
            with torch.no_grad():
                with autocast():
                    model(batch)
        except ValueError:
            pass  # Catcher stops execution after first layer
    print(f"Before restoring: {type(layers[0])}")
    layers[0] = original_layer
    del original_layer
    print(f"After restoring: {type(layers[0])}")
    assert not isinstance(layers[0], Catcher), "Catcher is still present!"
    torch.cuda.empty_cache()  # Free memory

    # Process layers sequentially with minimal memory usage
    attention_mask = cache['attention_mask']

    with torch.no_grad():  # Disable gradient tracking to save memory
        with autocast():
            for i in range(len(layers)):
                print(f"Processing Layer {i}")

                layer = layers[i].to(dev)  # Load one layer to GPU

                for j in range(nsamples):

                    inps[j] = layer(inps[j].unsqueeze(0).to(dev), attention_mask=attention_mask)[0].to(torch.device('cpu'))  # Process in-place

                layers[i] = layer.cpu()  # Move layer back to CPU
                del layer
                torch.cuda.empty_cache()  # Free up GPU memory

    # Move final components to GPU
    if model.model.decoder.final_layer_norm is not None:
        model.model.decoder.final_layer_norm = model.model.decoder.final_layer_norm.to(dev)
    if model.model.decoder.project_out is not None:
        model.model.decoder.project_out = model.model.decoder.project_out.to(dev)
    model.lm_head = model.lm_head.to(dev)

    testenc = testenc.to(dev)
    nlls = []

    # Compute perplexity
    with torch.no_grad():
        with autocast():
            for i in range(nsamples):
                print(f'sample {i}')
                hidden_states = inps[i].unsqueeze(0).to(dev)

                if model.model.decoder.final_layer_norm is not None:
                    hidden_states = model.model.decoder.final_layer_norm(hidden_states)
                if model.model.decoder.project_out is not None:
                    hidden_states = model.model.decoder.project_out(hidden_states)

                lm_logits = model.lm_head(hidden_states)
                del hidden_states
                shift_logits = lm_logits[:, :-1, :].contiguous()
                shift_labels = testenc[:, (i * mseq):((i + 1) * mseq)][:, 1:]

                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                del shift_logits
                neg_log_likelihood = loss.float() * mseq
                del loss
                nlls.append(neg_log_likelihood)

    ppl = torch.exp(torch.stack(nlls).sum() / (nsamples * mseq))
    print(f"Perplexity: {ppl.item():.3f}")

    if log_wandb:
        wandb.log({f'{dataset}/perplexity': ppl.item()})

    # Restore model's original config
    model.config.use_cache = use_cache



In [ ]:
from peft import get_peft_model_state_dict

# This will give you just the LoRA parameters
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        del param.grad


In [ ]:
model=model.to(torch.device('cpu'))
torch.cuda.empty_cache()

In [ ]:
model2=model.base_model.model

In [ ]:
print(torch.cuda.device_count())

In [ ]:
from torch.cuda.amp import autocast


In [ ]:
# # name="facebook/opt-125m"
# for dataset1 in ['ptb']:
#     trainenc2, testenc2 = get_loaders(
#         dataset1, seed=0, model="facebook/opt-125m", seqlen=2048
#     )
#     print(dataset1)

In [ ]:
import torch

total = torch.cuda.get_device_properties(0).total_memory
allocated = torch.cuda.memory_allocated(0)
cached = torch.cuda.memory_reserved(0)
available = total - allocated

print(f"Total:     {total / 1024**2:.2f} MB")
print(f"Allocated: {allocated / 1024**2:.2f} MB")
print(f"Cached:    {cached / 1024**2:.2f} MB")
print(f"Available: {available / 1024**2:.2f} MB (approx)")



In [ ]:
model.eval()

DEV = torch.device('cuda:0')

# testloader=trainenc2 #for ptb evaluation
testloader=testenc #for wikitext2 evaluation

opt_eval(model2, testloader, DEV, dataset)

## Share adapters on the 🤗 Hub

In [ ]:
# model.push_to_hub("samwit/bloom-7b1-lora-tagger",
#                   use_auth_token=True,
#                   commit_message="basic training",
#                   private=True)

## Load adapters from the Hub

In [ ]:
# import torch
# from peft import PeftModel, PeftConfig
# from transformers import AutoModelForCausalLM, AutoTokenizer

# peft_model_id = "samwit/bloom-7b1-lora-tagger"
# config = PeftConfig.from_pretrained(peft_model_id)
# model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, return_dict=True, load_in_8bit=True, device_map='auto')
# tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# # Load the Lora model
# model = PeftModel.from_pretrained(model, peft_model_id)

## Inference

In [ ]:
batch = tokenizer("“Training models with PEFT and LoRa is cool” ->: ", return_tensors='pt')

with torch.cuda.amp.autocast():
  output_tokens = model.generate(**batch, max_new_tokens=50)

print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))